# Running on an OQTOPUS device with `OqtopusEngine`

`OqtopusEngine` is OpenQARP's remote engine: it compiles every circuit for an [OQTOPUS](https://github.com/oqtopus-team) device itself and submits one OpenQASM 3 sampling job per circuit. The primitives, VQE and the shot-based gradients run on it unchanged.

The engine reaches the service through a small *transport*. `RemoteOqtopusTransport` (the default, behind the `[oqtopus]` extra) talks to a real deployment; `LocalOqtopusTransport` executes the very same programs on the local simulator with no credential, which is what this notebook uses. Everything below runs identically on the real transport.

In [ ]:
from qarp.algorithms import PauliAveraging, Sampler
from qarp.blocks import SimpleBlock
from qarp.engines import LocalOqtopusTransport, OqtopusEngine
from qarp.operators import QubitOperator

## A device

A transport serves the device's `device_info`: its qubits and its directed couplings. The three-qubit line below is the shape the OQTOPUS device-gateway publishes (`id`, `physical_id`, `couplings[].control/target`); the engine turns it into a `qarp.devices.Device` with the hardware basis `{x, sx, rz, cx}`.

In [ ]:
device_info = {
    "device_id": "line3",
    "qubits": [
        {"id": 0, "physical_id": 0},
        {"id": 1, "physical_id": 2},
        {"id": 2, "physical_id": 3},
    ],
    "couplings": [
        {"control": 0, "target": 1},
        {"control": 1, "target": 0},
        {"control": 1, "target": 2},
        {"control": 2, "target": 1},
    ],
}

transport = LocalOqtopusTransport(device_info=device_info, seed=1)
engine = OqtopusEngine("line3", transport=transport, n_shots=4000)
print(engine.device.n_qubits, engine.device.gate_set.name, sorted(engine.device.architecture.edges))

## Sampling a GHZ state

Qubits 0 and 2 are not coupled, so the engine routes the circuit, lowers it to the hardware basis and lays the measurements on the routed register. The counts come back in logical qubit order.

In [ ]:
ghz = SimpleBlock(3, name="ghz")
ghz.h(0).cx(0, 2).cx(2, 1)
ghz.build()

sampler = Sampler(ket=ghz)
engine.build([sampler])
print(engine.run()[0])

The transport logs every call, so we can look at the program the device received: only `x`, `sx`, `rz`, `cx` and one measurement per logical qubit.

In [ ]:
print(transport.log[-2]["program"])

## One VQE step

An expectation value is a `PauliAveraging` primitive: one job per commuting group of the operator. `batch_run` submits every job of a sweep before waiting, and `run_gradient` with the parameter-shift rule is one such sweep, so a gradient costs a single round trip to the device.

In [ ]:
import qarpx as qx

ansatz = SimpleBlock(2, name="ansatz")
ansatz.ry(0, qx.Param.symbol("a")).cx(0, 1).ry(1, qx.Param.symbol("b"))
ansatz.build()

hamiltonian = QubitOperator("Z0 Z1", 1.0) + QubitOperator("X0", 0.5) + QubitOperator("X1", 0.5)
energy = PauliAveraging(operator=hamiltonian, ket=ansatz)
engine.build([energy])

params = {"a": 0.4, "b": -0.3}
print("E(a, b) =", engine.run(params)[0])
print("dE/d(a, b) =", engine.run_gradient(params, method="parameter-shift")[0])
print("jobs submitted so far:", sum(entry["call"] == "submit" for entry in transport.log))

## The real transport

Against a deployment, drop the transport argument: `RemoteOqtopusTransport()` is built for you, and the SDK resolves the URL and token from `OQTOPUS_URL` / `OQTOPUS_API_TOKEN` or its config file. Nothing else changes.

```python
from qarp.engines import OqtopusEngine

engine = OqtopusEngine("qulacs", n_shots=4000)   # device id as the service names it
engine.build([sampler])
engine.run()
```

What the device cannot do — amplitudes, exact readout, an injected initial state, feed-forward — is refused with `CapabilityError` before anything is submitted, and a job the service fails raises `RemoteJobError` with the job id and the service message.